# Self-Healing Feeder: active-grid study with a harder IEEE 34-bus sweep (PLAN.md Addendum B2)

Same as `kaggle_active_grid.ipynb`, but the 34-bus SLG resistances are extended by 20 kOhm and 30 kOhm
(30 kOhm = R*, the smallest tested resistance whose fault current is <= 1.0% of the 34-bus head current,
the 123-bus 400 Ohm level; `results/transfer/signal_to_load.json`). Results go to `results/active_grid_hard34/`.

Simulation only. Nothing here is field data.

In [ ]:
import os, shutil, zipfile

DATASET_DIR = "/kaggle/input"
WORK = "/kaggle/working/self-healing-feeder"
if os.path.exists(WORK):
    shutil.rmtree(WORK)

src = next((root for root, _dirs, files in os.walk(DATASET_DIR) if "run.py" in files), None)
if src is None:
    zips = [os.path.join(r, f) for r, _d, fs in os.walk(DATASET_DIR) for f in fs if f.endswith(".zip")]
    for z in zips:
        zipfile.ZipFile(z).extractall("/kaggle/working/_bundle")
    src = next((root for root, _dirs, files in os.walk("/kaggle/working/_bundle") if "run.py" in files), None)
if src is None:
    raise SystemExit("Could not find project bundle in /kaggle/input")
shutil.copytree(src, WORK)
shutil.rmtree("/kaggle/working/_bundle", ignore_errors=True)
os.chdir(WORK)
print("Working dir:", WORK)
print("Top-level:", sorted(os.listdir(WORK)))

In [ ]:
# Install OpenDSS offline from bundled Linux wheels (no Internet required).
import subprocess, sys
if os.path.isdir(os.path.join(WORK, "wheels")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index",
                    "--find-links", "wheels", "OpenDSSDirect.py==0.9.4"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "OpenDSSDirect.py==0.9.4"], check=True)

import torch, opendssdirect as dss
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("OpenDSS:", dss.Basic.Version())

In [ ]:
# Fast verification tests on Kaggle environment before running the full experiment.
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_active_grid.py"], check=True)

In [ ]:
# Run the full study with the harder 34-bus sweep (Addendum B2).
subprocess.run([sys.executable, "-u", "scripts/active_grid_run.py", "--extra-r34", "20000,30000"], check=True)

In [ ]:
# Package results/active_grid_hard34, its run record and the probabilities into /kaggle/working.
import json
from pathlib import Path

OUT = Path("/kaggle/working/hard34_results")
if OUT.exists():
    shutil.rmtree(OUT)
(OUT / "active_grid_hard34").mkdir(parents=True)
(OUT / "provenance").mkdir(parents=True)
(OUT / "generated").mkdir(parents=True)
for p in sorted((Path(WORK) / "results" / "active_grid_hard34").glob("*.json")):
    shutil.copy2(p, OUT / "active_grid_hard34" / p.name)
shutil.copy2(Path(WORK) / "results" / "provenance" / "active_grid_hard34.json", OUT / "provenance" / "active_grid_hard34.json")
shutil.copy2(Path(WORK) / "data" / "generated" / "active_grid_hard34_probs.npz", OUT / "generated" / "active_grid_hard34_probs.npz")

os.chdir("/kaggle/working")
shutil.rmtree(WORK)
shutil.make_archive("/kaggle/working/hard34_results", "zip", OUT)
print("Saved /kaggle/working/hard34_results.zip")